# 03 — Text Mining (Path A: spaCy NER + sentence-transformers + zero-shot + BERTopic)

Turns the five free-text columns into structured signal — the highest-value step, since NB02 showed the
recoverable affluence signal lives in text, not amenity counts.

- **NER (spaCy):** named employers from "Nearby employment hubs"
- **Sentence embeddings (all-MiniLM-L6-v2):** 384-d vectors per text column + a combined locality vector
- **Zero-shot sector tagging (distilbart-mnli):** IT / Finance / Consulting / Manufacturing / Govt / Retail / Healthcare
- **Topic modelling (BERTopic):** themes across locality introductions
- **Regex-derived flags:** metro connectivity, airport minutes, maturity (emerging vs established)

Reads `artifacts/features_base.parquet`; writes `artifacts/features_text.parquet` + `artifacts/embeddings.npz`.

In [1]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

ART = Path.cwd() / "artifacts"
df = pd.read_parquet(ART / "features_base.parquet")
print("Loaded:", df.shape)

INTRO = "Locality introduction and neighbourhood"
EMP = "Nearby employment hubs"
SOCIAL = "Social & retail infra"
PHYS = "Physical infrastructure"
TEXT_COLS = [INTRO, EMP, SOCIAL, PHYS]

def txt(col):
    return df[col].fillna("").astype(str)

print({c: int(df[c].notna().sum()) for c in TEXT_COLS})

Loaded: (1001, 45)
{'Locality introduction and neighbourhood': 804, 'Nearby employment hubs': 804, 'Social & retail infra': 804, 'Physical infrastructure': 804}


In [2]:
# --- spaCy NER: extract named employers (ORG) from the employment column ---
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as spacy_download
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def employers(text):
    if not text:
        return []
    doc = nlp(text)
    orgs = [e.text.strip() for e in doc.ents if e.label_ == "ORG"]
    # de-dup, drop very short tokens
    seen, out = set(), []
    for o in orgs:
        k = o.lower()
        if len(o) > 2 and k not in seen:
            seen.add(k); out.append(o)
    return out

emp_lists = txt(EMP).apply(employers)
df["num_employers"] = emp_lists.apply(len)
df["top_employers"] = emp_lists.apply(lambda xs: ", ".join(xs[:5]) if xs else np.nan)
print("Localities with >=1 named employer:", int((df["num_employers"] > 0).sum()))
print(df.loc[df["num_employers"] > 0, ["AREA", "num_employers", "top_employers"]].head(5).to_string(index=False))

Localities with >=1 named employer: 786
                             AREA  num_employers                                                top_employers
              Attibele, Bangalore              1                                                   JP IT Park
            BTM Layout, Bangalore              3               Infosys, IBM, Prestige Blue Chip Software Park
             Banaswadi, Bangalore              8          ITPL, ITC Green Centre, RMZ Infinity Park, MNC, HCL
Bannerghatta Main Road, Bangalore              2              Mahindra & Mahindra, Federal-Mogul Goetze India
          Basavanagudi, Bangalore              6 Enzyme Tech Park, InsZoom Pvt Ltd., Oracle, NTT Data, VMware


In [3]:
# --- Sentence embeddings: per text column + combined locality vector ---
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(series):
    return embedder.encode(series.tolist(), batch_size=64, show_progress_bar=False,
                           normalize_embeddings=True).astype("float32")

emb_intro = embed(txt(INTRO))
emb_emp = embed(txt(EMP))
emb_social = embed(txt(SOCIAL))
emb_phys = embed(txt(PHYS))
combined_text = (txt(INTRO) + " " + txt(EMP) + " " + txt(SOCIAL) + " " + txt(PHYS)).str.strip()
emb_combined = embed(combined_text)

np.savez(ART / "embeddings.npz", area=df["AREA"].values, combined=emb_combined,
         intro=emb_intro, employment=emb_emp, social=emb_social, physical=emb_phys)
print("Embeddings:", emb_combined.shape, "saved to embeddings.npz")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3389.06it/s]

Embeddings: (1001, 384) saved to embeddings.npz


In [4]:
# --- Zero-shot sector tagging (transformers, distilbart-mnli) ---
from transformers import pipeline
zshot = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1", device=-1)

SECTORS = ["Information Technology", "Finance and Banking", "Consulting",
           "Manufacturing and Industry", "Government and Public Sector",
           "Retail and Commercial", "Healthcare and Pharma"]
SECTOR_KEY = {s: "sector_" + re.sub(r"[^a-z]+", "_", s.lower()).strip("_") for s in SECTORS}

emp_text = txt(EMP)
mask = emp_text.str.len() > 0
results = {}
seqs = emp_text[mask].tolist()
out = zshot(seqs, candidate_labels=SECTORS, multi_label=True, batch_size=8)
out = out if isinstance(out, list) else [out]

# initialise score columns (object dtype for the text label, float for scores)
for k in SECTOR_KEY.values():
    df[k] = np.nan
df["primary_sector"] = pd.Series(pd.NA, index=df.index, dtype="object")
df["primary_sector_score"] = np.nan

idxs = df.index[mask].tolist()
for i, res in zip(idxs, out):
    scores = dict(zip(res["labels"], res["scores"]))
    for s in SECTORS:
        df.at[i, SECTOR_KEY[s]] = round(float(scores.get(s, 0.0)), 4)
    top = max(scores, key=scores.get)
    df.at[i, "primary_sector"] = top
    df.at[i, "primary_sector_score"] = round(float(scores[top]), 4)

print("Tagged sectors for", int(mask.sum()), "localities")
print(df["primary_sector"].value_counts(dropna=True).to_string())

Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 231/231 [00:00<00:00, 27572.12it/s]

Tagged sectors for 804 localities
primary_sector
Information Technology          542
Manufacturing and Industry      142
Retail and Commercial            51
Government and Public Sector     43
Consulting                       14
Healthcare and Pharma             9
Finance and Banking               3


In [5]:
# --- Topic modelling on locality introductions (BERTopic, reusing the embedder) ---
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

intro_mask = df[INTRO].notna()
intro_docs = df.loc[intro_mask, INTRO].astype(str).tolist()
intro_emb = emb_intro[intro_mask.values]

# Stopword-filtered, bigram-aware vectorizer so topic keywords are real themes, not 'the/is/of'
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3)
topic_model = BERTopic(embedding_model=embedder, vectorizer_model=vectorizer,
                       min_topic_size=15, verbose=False)
topics, _ = topic_model.fit_transform(intro_docs, embeddings=intro_emb)

# map topic id -> short keyword label
def topic_label(tid):
    if tid == -1:
        return "misc"
    words = [w for w, _ in topic_model.get_topic(tid)[:4]]
    return ", ".join(words)

df["topic_id"] = np.nan
df["topic_keywords"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[intro_mask, "topic_id"] = topics
df.loc[intro_mask, "topic_keywords"] = [topic_label(t) for t in topics]

print("Topics found:", len([t for t in set(topics) if t != -1]),
      "| outliers:", int((np.array(topics) == -1).sum()))
print(topic_model.get_topic_info().head(8)[["Topic", "Count", "Name"]].to_string(index=False))

Topics found: 3 | outliers: 0
 Topic  Count                              Name
     0    669 0_residential_locality_nagar_road
     1    112 1_sector_gurgaon_road_residential
     2     23    2_dwarka_delhi_sector_locality


In [6]:
# --- Regex-derived flags from physical infra + intro ---
def metro_connected(text):
    return bool(re.search(r"metro", str(text), re.I)) if pd.notna(text) else False

def airport_min(text):
    if pd.isna(text):
        return np.nan
    m = re.search(r"airport.{0,60}?(\d{1,3})\s*[-]\s*(\d{1,3})\s*minutes", str(text), re.I)
    if m:
        return int(m.group(2))
    if re.search(r"airport", str(text), re.I):
        mins = [int(x) for x in re.findall(r"(\d{1,3})\s*minutes", str(text))]
        return max(mins) if mins else np.nan
    return np.nan

EMERGING = r"emerging|developing|upcoming|under[- ]construction|nascent"
ESTABLISHED = r"established|well[- ]developed|posh|premium|sought[- ]after|prominent"

def maturity(text):
    if pd.isna(text):
        return np.nan
    t = str(text).lower()
    e, s = bool(re.search(EMERGING, t)), bool(re.search(ESTABLISHED, t))
    if e and not s: return "emerging"
    if s and not e: return "established"
    if s and e: return "mixed"
    return np.nan

df["is_metro_connected"] = df[PHYS].apply(metro_connected)
df["airport_min"] = df[PHYS].apply(airport_min)
df["maturity"] = df[INTRO].apply(maturity)
print("metro connected:", df["is_metro_connected"].value_counts(dropna=False).to_dict())
print("maturity:", df["maturity"].value_counts(dropna=False).to_dict())

metro connected: {True: 621, False: 380}
maturity: {nan: 582, 'established': 261, 'emerging': 129, 'mixed': 29}


In [7]:
# --- Persist ---
out = ART / "features_text.parquet"
df.to_parquet(out, index=False)
print("Saved", df.shape[0], "rows x", df.shape[1], "cols ->", out.relative_to(Path.cwd()))
new = ["num_employers", "top_employers", "primary_sector", "primary_sector_score",
       "topic_id", "topic_keywords", "is_metro_connected", "airport_min", "maturity"]
print("New text features:", new)

Saved 1001 rows x 61 cols -> artifacts\features_text.parquet
New text features: ['num_employers', 'top_employers', 'primary_sector', 'primary_sector_score', 'topic_id', 'topic_keywords', 'is_metro_connected', 'airport_min', 'maturity']


## Output
`artifacts/features_text.parquet` + `artifacts/embeddings.npz` — named employers, zero-shot sector
profile, locality topics, semantic embeddings, and connectivity/maturity flags.

**Next:** `04_geo_graph.ipynb` — spatial clustering + the locality adjacency graph (centrality, Louvain belts).